In [1]:
from pathlib import Path
import pandas as pd

DATASET_DIR = Path(
    "/kaggle/input/datasets/kevinardhana/"
    "odir-5k-patient-level-multi-label-fundus-dataset"
)

IMAGE_DIR = DATASET_DIR / "Training Images" / "Training Images"

TRAIN_CSV = DATASET_DIR / "train.csv"
VALID_CSV = DATASET_DIR / "validation.csv"
TEST_CSV = DATASET_DIR / "test.csv"

LABELS = ["N", "D", "G", "C", "A", "H", "M", "O"]

train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train      :", train_df.shape)
print("Validation :", valid_df.shape)
print("Test       :", test_df.shape)
print("\nKolom:")
print(train_df.columns.tolist())

assert len(train_df) == 2450
assert len(valid_df) == 525
assert len(test_df) == 525
assert all(column in train_df.columns for column in LABELS)

print("\nContoh data:")
display(train_df.head())

print("\nCitra kiri contoh ada :", (IMAGE_DIR / train_df.loc[0, "left_image"]).is_file())
print("Citra kanan contoh ada:", (IMAGE_DIR / train_df.loc[0, "right_image"]).is_file())

Train      : (2450, 16)
Validation : (525, 16)
Test       : (525, 16)

Kolom:
['patient_id', 'left_image', 'right_image', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O', 'label_count', 'is_multilabel', 'left_exists', 'right_exists', 'split']

Contoh data:


,patient_id,left_image,right_image,N,D,G,C,A,H,M,O,label_count,is_multilabel,left_exists,right_exists,split
0,1,1_left.jpg,1_right.jpg,1,0,0,0,0,0,0,0,1,0,True,True,train
1,3,3_left.jpg,3_right.jpg,0,0,0,0,0,0,0,1,1,0,True,True,train
2,4,4_left.jpg,4_right.jpg,0,1,0,0,0,0,0,1,2,1,True,True,train
3,5,5_left.jpg,5_right.jpg,0,1,0,0,0,0,0,0,1,0,True,True,train
4,6,6_left.jpg,6_right.jpg,0,1,0,0,0,0,0,1,2,1,True,True,train



Citra kiri contoh ada : True
Citra kanan contoh ada: True


In [2]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

for path in INPUT_ROOT.rglob("train.csv"):
    print("train.csv ditemukan di:", path)

for path in INPUT_ROOT.rglob("0_left.jpg"):
    print("Contoh citra ditemukan di:", path)

train.csv ditemukan di: /kaggle/input/datasets/kevinardhana/odir-5k-patient-level-multi-label-fundus-dataset/train.csv
Contoh citra ditemukan di: /kaggle/input/datasets/kevinardhana/odir-5k-patient-level-multi-label-fundus-dataset/Training Images/Training Images/0_left.jpg


In [3]:
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

SEED = 42
BATCH_SIZE = 16
NUM_WORKERS = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


class FundusPairDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        left_path = self.image_dir / row["left_image"]
        right_path = self.image_dir / row["right_image"]

        left_image = Image.open(left_path).convert("RGB")
        right_image = Image.open(right_path).convert("RGB")

        left_image = self.transform(left_image)
        right_image = self.transform(right_image)

        labels = torch.tensor(
            row[LABELS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "left_image": left_image,
            "right_image": right_image,
            "labels": labels,
            "patient_id": int(row["patient_id"]),
        }


train_dataset = FundusPairDataset(train_df, IMAGE_DIR, train_transform)
valid_dataset = FundusPairDataset(valid_df, IMAGE_DIR, eval_transform)
test_dataset = FundusPairDataset(test_df, IMAGE_DIR, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

batch = next(iter(train_loader))

print("Citra kiri :", batch["left_image"].shape)
print("Citra kanan:", batch["right_image"].shape)
print("Label      :", batch["labels"].shape)
print("Contoh target:", batch["labels"][0].tolist())

Citra kiri : torch.Size([16, 3, 224, 224])
Citra kanan: torch.Size([16, 3, 224, 224])
Label      : torch.Size([16, 8])
Contoh target: [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [4]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


class SharedResNet50(nn.Module):
    def __init__(self, num_labels=8, dropout=0.30):
        super().__init__()

        weights = ResNet50_Weights.IMAGENET1K_V2
        self.backbone = resnet50(weights=weights)

        feature_dim = self.backbone.fc.in_features  # 2.048 fitur
        self.backbone.fc = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(feature_dim * 2, num_labels)
        )

    def forward(self, left_image, right_image):
        left_features = self.backbone(left_image)
        right_features = self.backbone(right_image)

        fused_features = torch.cat(
            [left_features, right_features],
            dim=1
        )

        logits = self.classifier(fused_features)
        return logits


model = SharedResNet50(num_labels=len(LABELS)).to(DEVICE)

left_batch = batch["left_image"].to(DEVICE)
right_batch = batch["right_image"].to(DEVICE)

with torch.no_grad():
    logits = model(left_batch, right_batch)
    probabilities = torch.sigmoid(logits)

print("Logits shape       :", logits.shape)
print("Probabilities shape:", probabilities.shape)
print("Contoh probabilitas:", probabilities[0].cpu().numpy())

Device: cuda
GPU: Tesla T4
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 212MB/s]


Logits shape       : torch.Size([16, 8])
Probabilities shape: torch.Size([16, 8])
Contoh probabilitas: [0.5393931  0.49760547 0.43605843 0.49674216 0.50723016 0.5654205
 0.52663934 0.50693965]


In [5]:
import json
from pathlib import Path
from sklearn.metrics import f1_score
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 7

OUTPUT_DIR = Path("/kaggle/working/bce_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

criterion = nn.BCEWithLogitsLoss()

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch in loader:
        left_images = batch["left_image"].to(device)
        right_images = batch["right_image"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(left_images, right_images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.5):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    all_targets = []
    all_probabilities = []

    for batch in loader:
        left_images = batch["left_image"].to(device)
        right_images = batch["right_image"].to(device)
        labels = batch["labels"].to(device)

        logits = model(left_images, right_images)
        loss = criterion(logits, labels)
        probabilities = torch.sigmoid(logits)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_targets.append(labels.cpu().numpy())
        all_probabilities.append(probabilities.cpu().numpy())

    y_true = np.concatenate(all_targets)
    y_prob = np.concatenate(all_probabilities)
    y_pred = (y_prob >= threshold).astype(int)

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return {
        "loss": total_loss / total_samples,
        "macro_f1": macro_f1,
        "y_true": y_true,
        "y_prob": y_prob,
    }


print("BCE, AdamW, scheduler, dan fungsi training siap.")

BCE, AdamW, scheduler, dan fungsi training siap.


In [6]:
history = []
best_macro_f1 = -1.0
best_epoch = 0
patience_counter = 0

checkpoint_path = OUTPUT_DIR / "best_bce_resnet50.pt"
history_path = OUTPUT_DIR / "training_history.json"

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE
    )

    validation_result = evaluate(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=DEVICE,
        threshold=0.5
    )

    validation_loss = validation_result["loss"]
    validation_macro_f1 = validation_result["macro_f1"]

    scheduler.step(validation_macro_f1)

    current_lr = optimizer.param_groups[0]["lr"]

    epoch_result = {
        "epoch": epoch,
        "train_loss": float(train_loss),
        "validation_loss": float(validation_loss),
        "validation_macro_f1": float(validation_macro_f1),
        "learning_rate": float(current_lr),
    }
    history.append(epoch_result)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train loss: {train_loss:.4f} | "
        f"val loss: {validation_loss:.4f} | "
        f"val Macro-F1: {validation_macro_f1:.4f} | "
        f"lr: {current_lr:.6f}"
    )

    if validation_macro_f1 > best_macro_f1:
        best_macro_f1 = validation_macro_f1
        best_epoch = epoch
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "validation_macro_f1": best_macro_f1,
                "labels": LABELS,
                "threshold": 0.5,
                "backbone": "ResNet50",
                "loss_function": "BCEWithLogitsLoss",
            },
            checkpoint_path
        )

        print("  ✓ Checkpoint terbaik disimpan.")

    else:
        patience_counter += 1

    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping pada epoch {epoch}. "
            f"Checkpoint terbaik: epoch {best_epoch}."
        )
        break


with history_path.open("w") as file:
    json.dump(history, file, indent=2)

print("\nPelatihan selesai.")
print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", round(best_macro_f1, 4))
print("Checkpoint:", checkpoint_path)

Epoch 01/30 | train loss: 0.3532 | val loss: 0.3088 | val Macro-F1: 0.2358 | lr: 0.000100
  ✓ Checkpoint terbaik disimpan.
Epoch 02/30 | train loss: 0.2901 | val loss: 0.3098 | val Macro-F1: 0.3127 | lr: 0.000100
  ✓ Checkpoint terbaik disimpan.
Epoch 03/30 | train loss: 0.2694 | val loss: 0.2883 | val Macro-F1: 0.3579 | lr: 0.000100
  ✓ Checkpoint terbaik disimpan.
Epoch 04/30 | train loss: 0.2496 | val loss: 0.2804 | val Macro-F1: 0.4335 | lr: 0.000100
  ✓ Checkpoint terbaik disimpan.
Epoch 05/30 | train loss: 0.2317 | val loss: 0.2832 | val Macro-F1: 0.4045 | lr: 0.000100
Epoch 06/30 | train loss: 0.2123 | val loss: 0.2906 | val Macro-F1: 0.4687 | lr: 0.000100
  ✓ Checkpoint terbaik disimpan.
Epoch 07/30 | train loss: 0.1958 | val loss: 0.2983 | val Macro-F1: 0.5009 | lr: 0.000100
  ✓ Checkpoint terbaik disimpan.
Epoch 08/30 | train loss: 0.1773 | val loss: 0.2835 | val Macro-F1: 0.5246 | lr: 0.000100
  ✓ Checkpoint terbaik disimpan.
Epoch 09/30 | train loss: 0.1526 | val loss: 0.31

In [7]:
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/bce_baseline")
checkpoint_path = OUTPUT_DIR / "best_bce_resnet50.pt"

print("Lokasi checkpoint:", checkpoint_path)
print("Checkpoint tersedia:", checkpoint_path.exists())

Lokasi checkpoint: /kaggle/working/bce_baseline/best_bce_resnet50.pt
Checkpoint tersedia: True


In [8]:
# Memuat checkpoint terbaik dan menentukan threshold BCE
# menggunakan validation set

from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet50


LABELS = ["N", "D", "G", "C", "A", "H", "M", "O"]

BATCH_SIZE = 16
NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# Lokasi dataset dan hasil training
DATASET_DIR = Path(
    "/kaggle/input/datasets/kevinardhana/"
    "odir-5k-patient-level-multi-label-fundus-dataset"
)

IMAGE_DIR = (
    DATASET_DIR
    / "Training Images"
    / "Training Images"
)

VALID_CSV = DATASET_DIR / "validation.csv"

OUTPUT_DIR = Path(
    "/kaggle/working/bce_baseline"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Dataset pasangan citra mata kiri dan kanan
class FundusPairDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        transform
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        left_path = (
            self.image_dir
            / row["left_image"]
        )

        right_path = (
            self.image_dir
            / row["right_image"]
        )

        left_image = Image.open(
            left_path
        ).convert("RGB")

        right_image = Image.open(
            right_path
        ).convert("RGB")

        left_image = self.transform(
            left_image
        )

        right_image = self.transform(
            right_image
        )

        labels = torch.tensor(
            row[LABELS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "left_image": left_image,
            "right_image": right_image,
            "labels": labels
        }


# Arsitektur harus sama dengan model saat training
class SharedResNet50(nn.Module):
    def __init__(
        self,
        num_labels=8,
        dropout=0.30
    ):
        super().__init__()

        # Tidak mengunduh bobot ImageNet karena bobot model
        # akan dimuat dari checkpoint hasil training.
        self.backbone = resnet50(
            weights=None
        )

        feature_dim = (
            self.backbone.fc.in_features
        )

        self.backbone.fc = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(
                feature_dim * 2,
                num_labels
            )
        )

    def forward(
        self,
        left_image,
        right_image
    ):
        left_features = self.backbone(
            left_image
        )

        right_features = self.backbone(
            right_image
        )

        combined_features = torch.cat(
            [
                left_features,
                right_features
            ],
            dim=1
        )

        return self.classifier(
            combined_features
        )


@torch.no_grad()
def collect_predictions(
    model,
    loader,
    device
):
    model.eval()

    all_targets = []
    all_probabilities = []

    for batch in loader:
        left_images = batch["left_image"].to(
            device,
            non_blocking=True
        )

        right_images = batch["right_image"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        logits = model(
            left_images,
            right_images
        )

        probabilities = torch.sigmoid(
            logits
        )

        all_targets.append(
            labels.cpu().numpy()
        )

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

    targets = np.concatenate(
        all_targets,
        axis=0
    )

    probabilities = np.concatenate(
        all_probabilities,
        axis=0
    )

    return targets, probabilities


# Transformasi validation sama dengan baseline
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Membaca validation set
valid_df = pd.read_csv(
    VALID_CSV
)

assert len(valid_df) == 525, (
    f"Jumlah validation tidak sesuai: {len(valid_df)}"
)

assert all(
    label in valid_df.columns
    for label in LABELS
), "Kolom label validation tidak lengkap"


valid_dataset = FundusPairDataset(
    dataframe=valid_df,
    image_dir=IMAGE_DIR,
    transform=eval_transform
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda"
)


# Checkpoint dibuat oleh cell training sebelumnya
checkpoint_path = (
    OUTPUT_DIR
    / "best_bce_resnet50.pt"
)

assert checkpoint_path.is_file(), (
    "Checkpoint tidak ditemukan. "
    f"Lokasi yang diperiksa: {checkpoint_path}"
)


checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)


required_checkpoint_keys = {
    "epoch",
    "model_state_dict"
}

missing_keys = (
    required_checkpoint_keys
    - set(checkpoint.keys())
)

assert not missing_keys, (
    "Isi checkpoint tidak lengkap. "
    f"Key yang tidak ditemukan: {missing_keys}"
)


# Membuat model dan memuat bobot terbaik
model = SharedResNet50(
    num_labels=len(LABELS),
    dropout=0.30
).to(DEVICE)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()


print("Checkpoint ditemukan:", checkpoint_path)
print("Checkpoint epoch:", checkpoint["epoch"])
print("Device evaluasi:", DEVICE)


# Menghasilkan probabilitas validation
y_val, prob_val = collect_predictions(
    model=model,
    loader=valid_loader,
    device=DEVICE
)


assert y_val.shape == (525, 8), (
    f"Bentuk target validation tidak sesuai: {y_val.shape}"
)

assert prob_val.shape == (525, 8), (
    "Bentuk probabilitas validation "
    f"tidak sesuai: {prob_val.shape}"
)


# Mencari threshold terbaik untuk setiap label
candidate_thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

best_thresholds = []
threshold_rows = []


for label_index, label_name in enumerate(LABELS):
    label_scores = []

    for threshold in candidate_thresholds:
        label_predictions = (
            prob_val[:, label_index]
            >= threshold
        ).astype(np.int32)

        score = f1_score(
            y_val[:, label_index],
            label_predictions,
            zero_division=0
        )

        label_scores.append(score)

    best_position = int(
        np.argmax(label_scores)
    )

    selected_threshold = float(
        candidate_thresholds[best_position]
    )

    selected_f1 = float(
        label_scores[best_position]
    )

    best_thresholds.append(
        selected_threshold
    )

    threshold_rows.append({
        "label": label_name,
        "threshold_terbaik":
            selected_threshold,
        "F1_validation":
            selected_f1,
        "jumlah_positif_validation":
            int(y_val[:, label_index].sum())
    })


best_thresholds = np.asarray(
    best_thresholds,
    dtype=np.float32
)


# Membandingkan threshold 0,50 dengan threshold per label
default_predictions = (
    prob_val >= 0.50
).astype(np.int32)

optimized_predictions = (
    prob_val >= best_thresholds
).astype(np.int32)


macro_f1_default = f1_score(
    y_val,
    default_predictions,
    average="macro",
    zero_division=0
)

macro_f1_optimized = f1_score(
    y_val,
    optimized_predictions,
    average="macro",
    zero_division=0
)


threshold_df = pd.DataFrame(
    threshold_rows
)


print(
    "\nMacro-F1 threshold 0,50:",
    f"{macro_f1_default:.4f}"
)

print(
    "Macro-F1 threshold per label:",
    f"{macro_f1_optimized:.4f}"
)

print("\nThreshold setiap label:")
display(threshold_df)


# Menyimpan threshold
threshold_csv_path = (
    OUTPUT_DIR
    / "validation_thresholds_bce.csv"
)

threshold_json_path = (
    OUTPUT_DIR
    / "validation_thresholds_bce.json"
)


threshold_df.to_csv(
    threshold_csv_path,
    index=False
)


with open(
    threshold_json_path,
    "w"
) as file:
    json.dump(
        {
            "checkpoint_epoch":
                int(checkpoint["epoch"]),
            "labels":
                LABELS,
            "thresholds":
                best_thresholds.tolist(),
            "macro_f1_threshold_0_5":
                float(macro_f1_default),
            "macro_f1_optimized":
                float(macro_f1_optimized)
        },
        file,
        indent=2
    )


print("\nThreshold berhasil disimpan:")
print("-", threshold_csv_path)
print("-", threshold_json_path)

Checkpoint ditemukan: /kaggle/working/bce_baseline/best_bce_resnet50.pt
Checkpoint epoch: 27
Device evaluasi: cuda

Macro-F1 threshold 0,50: 0.5566
Macro-F1 threshold per label: 0.5943

Threshold setiap label:


,label,threshold_terbaik,F1_validation,jumlah_positif_validation
0,N,0.18,0.634615,171
1,D,0.09,0.594595,169
2,G,0.06,0.475000,32
3,C,0.46,0.888889,32
4,A,0.27,0.571429,25
5,H,0.31,0.210526,16
6,M,0.25,0.857143,26
7,O,0.11,0.521978,147



Threshold berhasil disimpan:
- /kaggle/working/bce_baseline/validation_thresholds_bce.csv
- /kaggle/working/bce_baseline/validation_thresholds_bce.json


In [9]:
from pathlib import Path
import json, numpy as np, pandas as pd, torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50
from sklearn.metrics import f1_score, roc_auc_score, hamming_loss, accuracy_score, precision_recall_fscore_support, multilabel_confusion_matrix

LABELS = ["N", "D", "G", "C", "A", "H", "M", "O"]
THRESHOLDS = np.asarray(best_thresholds, dtype=np.float32)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path("/kaggle/input/datasets/kevinardhana/odir-5k-patient-level-multi-label-fundus-dataset")
IMAGE_DIR = ROOT / "Training Images" / "Training Images"
OUTPUT_DIR = Path("/kaggle/working/bce_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class PairDataset(Dataset):
    def __init__(self, df, transform): self.df, self.transform = df.reset_index(drop=True), transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        left = Image.open(IMAGE_DIR / row.left_image).convert("RGB")
        right = Image.open(IMAGE_DIR / row.right_image).convert("RGB")
        return self.transform(left), self.transform(right), torch.tensor(row[LABELS].values.astype(np.float32))

class SharedResNet50(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = resnet50(weights=None)
        features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Sequential(nn.Dropout(0.30), nn.Linear(features * 2, len(LABELS)))
    def forward(self, left, right):
        return self.classifier(torch.cat([self.backbone(left), self.backbone(right)], dim=1))

checkpoint_path = OUTPUT_DIR / "best_bce_resnet50.pt"
assert checkpoint_path.is_file(), checkpoint_path
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model = SharedResNet50().to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

transform = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])])
test_df = pd.read_csv(ROOT / "test.csv")
loader = DataLoader(PairDataset(test_df, transform), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

y_true, probabilities = [], []
with torch.no_grad():
    for left, right, labels in loader:
        probabilities.append(torch.sigmoid(model(left.to(DEVICE), right.to(DEVICE))).cpu().numpy())
        y_true.append(labels.numpy())
y_true, probabilities = np.concatenate(y_true), np.concatenate(probabilities)
y_pred = (probabilities >= THRESHOLDS).astype(int)

precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
per_class = pd.DataFrame({"label": LABELS, "precision": precision, "recall": recall, "f1_score": f1, "auroc": roc_auc_score(y_true, probabilities, average=None), "support": support, "threshold": THRESHOLDS})
summary = {"checkpoint_epoch": int(checkpoint["epoch"]), "macro_f1": float(f1_score(y_true,y_pred,average="macro",zero_division=0)), "micro_f1": float(f1_score(y_true,y_pred,average="micro",zero_division=0)), "macro_auroc": float(roc_auc_score(y_true,probabilities,average="macro")), "hamming_loss": float(hamming_loss(y_true,y_pred)), "subset_accuracy": float(accuracy_score(y_true,y_pred))}
cm = multilabel_confusion_matrix(y_true,y_pred)
confusion = pd.DataFrame([{"label": label, "TN": int(x[0,0]), "FP": int(x[0,1]), "FN": int(x[1,0]), "TP": int(x[1,1])} for label,x in zip(LABELS,cm)])
predictions = pd.DataFrame({"patient_id": test_df.patient_id})
for i, label in enumerate(LABELS):
    predictions[f"true_{label}"] = y_true[:,i].astype(int); predictions[f"prob_{label}"] = probabilities[:,i]; predictions[f"pred_{label}"] = y_pred[:,i]
per_class.to_csv(OUTPUT_DIR / "test_per_class_metrics_bce.csv", index=False)
confusion.to_csv(OUTPUT_DIR / "test_confusion_matrix_bce.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "test_predictions_bce.csv", index=False)
with open(OUTPUT_DIR / "test_metrics_bce.json", "w") as f: json.dump(summary,f,indent=2)
print("Hasil final test set, BCE baseline")
for key,value in summary.items(): print(f"{key}: {value:.4f}" if isinstance(value,float) else f"{key}: {value}")
display(per_class)
display(confusion)


Hasil final test set, BCE baseline
checkpoint_epoch: 27
macro_f1: 0.5955
micro_f1: 0.5827
macro_auroc: 0.8544
hamming_loss: 0.1357
subset_accuracy: 0.3638


,label,precision,recall,f1_score,auroc,support,threshold
0,N,0.501916,0.757225,0.603687,0.776274,173,0.18
1,D,0.603896,0.550296,0.575851,0.774483,169,0.09
2,G,0.473684,0.562500,0.514286,0.872781,32,0.06
3,C,0.800000,0.750000,0.774194,0.947008,32,0.46
4,A,0.636364,0.583333,0.608696,0.926730,24,0.27
5,H,0.750000,0.200000,0.315789,0.809281,15,0.31
6,M,0.952381,0.769231,0.851064,0.996146,26,0.25
7,O,0.435780,0.646259,0.520548,0.732408,147,0.11


,label,TN,FP,FN,TP
0,N,222,130,42,131
1,D,295,61,76,93
2,G,473,20,14,18
3,C,487,6,8,24
4,A,493,8,10,14
5,H,509,1,12,3
6,M,498,1,6,20
7,O,255,123,52,95
